# MCPify your AWS Lambda with Gateway OAuth Inbound
## Transform AWS Lambda functions into secure MCP tools with Bedrock AgentCore Gateway

# 使用 Gateway OAuth 入站认证将您的 AWS Lambda MCP 化
## 使用 Bedrock AgentCore Gateway 将 AWS Lambda 函数转换为安全的 MCP 工具

## Overview

## 概述

Bedrock AgentCore Gateway provides customers a way to turn their existing AWS Lambda functions into fully-managed MCP servers without needing to manage infra or hosting. Gateway will provide a uniform Model Context Protocol (MCP) interface across all these tools. Gateway employs a dual authentication model to ensure secure access control for both incoming requests and outbound connections to target resources. The framework consists of two key components: Inbound Auth, which validates and authorizes users attempting to access gateway targets, and Outbound Auth, which enables the gateway to securely connect to backend resources on behalf of authenticated users. Gateways uses IAM role to authorize the calls to AWS Lambda functions for outbound authorization.

Bedrock AgentCore Gateway 为客户提供了一种将现有 AWS Lambda 函数转换为完全托管的 MCP 服务器的方式，无需管理基础设施或托管。Gateway 将为所有这些工具提供统一的模型上下文协议（MCP）接口。Gateway 采用双重身份验证模型，以确保对传入请求和到目标资源的出站连接进行安全访问控制。该框架由两个关键组件组成：入站认证（Inbound Auth），用于验证和授权尝试访问网关目标的用户；出站认证（Outbound Auth），使网关能够代表经过身份验证的用户安全地连接到后端资源。Gateway 使用 IAM 角色来授权对 AWS Lambda 函数的出站授权调用。

In this example, we will demonstrate OAuth for inbound authorization and IAM roles for outbound authorization.

在此示例中，我们将演示用于入站授权的 OAuth 和用于出站授权的 IAM 角色。

![How does it work](images/lambda-iam-gateway.png)

### Tutorial Details

### 教程详情

| Information          | Details                                                   |
|:---------------------|:----------------------------------------------------------|
| Tutorial type        | Interactive                                               |
| AgentCore components | AgentCore Gateway, AgentCore Identity                     |
| Agentic Framework    | Strands Agents                                            |
| Gateway Target type  | AWS Lambda                                                |
| Inbound Auth IdP     | Amazon Cognito                                            |
| Outbound Auth        | AWS IAM                                                   |
| LLM model            | Anthropic Claude Haiku 4.5, Amazon Nova Pro              |
| Tutorial components  | Creating AgentCore Gateway and Invoking AgentCore Gateway |
| Tutorial vertical    | Cross-vertical                                            |
| Example complexity   | Easy                                                      |
| SDK used             | boto3                                                     |

| 信息                  | 详情                                                       |
|:---------------------|:----------------------------------------------------------|
| 教程类型              | 交互式                                                     |
| AgentCore 组件       | AgentCore Gateway、AgentCore Identity                     |
| 代理框架              | Strands Agents                                            |
| Gateway 目标类型     | AWS Lambda                                                |
| 入站认证 IdP         | Amazon Cognito                                            |
| 出站认证              | AWS IAM                                                   |
| LLM 模型             | Anthropic Claude Haiku 4.5、Amazon Nova Pro               |
| 教程组件              | 创建 AgentCore Gateway 和调用 AgentCore Gateway            |
| 教程行业              | 跨行业                                                     |
| 示例复杂度            | 简单                                                       |
| 使用的 SDK           | boto3                                                     |

In the first part of the tutorial we will create some AmazonCore Gateway targets

在教程的第一部分，我们将创建一些 AmazonCore Gateway 目标

### Tutorial Architecture

### 教程架构

In this tutorial we will transform operations defined in AWS lambda function into MCP tools and host it in Bedrock AgentCore Gateway.
For demonstration purposes, we will use a Strands Agent using Amazon Bedrock models
In our example we will use a very simple agent with two tools: get_order and update_order.

在本教程中，我们将把 AWS Lambda 函数中定义的操作转换为 MCP 工具，并将其托管在 Bedrock AgentCore Gateway 中。
出于演示目的，我们将使用一个使用 Amazon Bedrock 模型的 Strands Agent
在我们的示例中，我们将使用一个具有两个工具的非常简单的代理：get_order 和 update_order。

## Prerequisites

## 前提条件

To execute this tutorial you will need:
* Jupyter notebook (Python kernel)
* uv
* AWS credentials
* Amazon Cognito

要执行本教程，您需要：
* Jupyter notebook（Python 内核）
* uv
* AWS 凭证
* Amazon Cognito

## Configuring Authentication for Incoming AgentCore Gateway Requests

## 为传入的 AgentCore Gateway 请求配置身份验证

AgentCore Gateway provides secure connections via inbound and outbound authentication. For the inbound authentication, the AgentCore Gateway analyzes the OAuth token passed during invocation to decide allow or deny the access to a tool in the gateway. If a tool needs access to external resources, the AgentCore Gateway can use outbound authentication via API Key, IAM or OAuth Token to allow or deny the access to the external resource.

AgentCore Gateway 通过入站和出站身份验证提供安全连接。对于入站身份验证，AgentCore Gateway 会分析调用期间传递的 OAuth 令牌，以决定允许或拒绝对网关中工具的访问。如果工具需要访问外部资源，AgentCore Gateway 可以使用通过 API Key、IAM 或 OAuth 令牌的出站身份验证来允许或拒绝对外部资源的访问。

During the inbound authorization flow, an agent or the MCP client calls an MCP tool in the AgentCore Gateway adding an OAuth access token (generated from the user's IdP). AgentCore Gateway then validates the OAuth access token and performs inbound authorization.

在入站授权流程中，代理或 MCP 客户端在 AgentCore Gateway 中调用 MCP 工具，并添加 OAuth 访问令牌（从用户的 IdP 生成）。然后 AgentCore Gateway 验证 OAuth 访问令牌并执行入站授权。

If the tool running in AgentCore Gateway needs to access external resources, OAuth will retrieve credentials of downstream resources using the resource credential provider for the Gateway target. AgentCore Gateway pass the authorization credentials to the caller to get access to the downstream API.

如果在 AgentCore Gateway 中运行的工具需要访问外部资源，OAuth 将使用 Gateway 目标的资源凭证提供程序检索下游资源的凭证。AgentCore Gateway 将授权凭证传递给调用者以获取对下游 API 的访问权限。

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1') # set the AWS region

In [ ]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

In [ ]:
#### Create a sample AWS Lambda function that you want to convert into MCP tools
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda function created with ARN: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda function creation failed with message: ", lambda_resp['lambda_function_arn'])

In [ ]:
#### Create an IAM role for the Gateway to assume
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

# Create Amazon Cognito Pool for Inbound authorization to Gateway

# 为 Gateway 入站授权创建 Amazon Cognito 用户池

In [ ]:
# Creating Cognito User Pool 
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret  = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# Get discovery URL  
cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

# Create the Gateway with Amazon Cognito Authorizer for inbound authorization

# 使用 Amazon Cognito 授权器创建 Gateway 用于入站授权

In [ ]:
# CreateGateway with Cognito authorizer without CMK. Use the Cognito user pool created in the previous step
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='TestGWforLambda',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AgentCore Gateway with AWS Lambda target type'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

# Create an AWS Lambda target and transform into MCP tools

# 创建 AWS Lambda 目标并转换为 MCP 工具

In [ ]:
# Replace the AWS Lambda function ARN below
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp['lambda_function_arn'], # Replace this with your AWS Lambda function ARN
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='Lambda Target using SDK',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Calling Bedrock AgentCore Gateway from a Strands Agent

# 从 Strands Agent 调用 Bedrock AgentCore Gateway

The Strands agent seamlessly integrates with AWS tools through the Bedrock AgentCore Gateway, which implements the Model Context Protocol (MCP) specification. This integration enables secure, standardized communication between AI agents and AWS services.

Strands 代理通过实现模型上下文协议（MCP）规范的 Bedrock AgentCore Gateway 与 AWS 工具无缝集成。此集成实现了 AI 代理与 AWS 服务之间的安全、标准化通信。

At its core, the Bedrock AgentCore Gateway serves as a protocol-compliant Gateway that exposes fundamental MCP APIs: ListTools and InvokeTools. These APIs allow any MCP-compliant client or SDK to discover and interact with available tools in a secure, standardized way. When the Strands agent needs to access AWS services, it communicates with the Gateway using these MCP-standardized endpoints.

从本质上讲，Bedrock AgentCore Gateway 作为符合协议的网关，公开基本的 MCP API：ListTools 和 InvokeTools。这些 API 允许任何符合 MCP 的客户端或 SDK 以安全、标准化的方式发现和与可用工具交互。当 Strands 代理需要访问 AWS 服务时，它使用这些 MCP 标准化端点与 Gateway 通信。

The Gateway's implementation adheres strictly to the (MCP Authorization specification)[https://modelcontextprotocol.org/specification/draft/basic/authorization], ensuring robust security and access control. This means that every tool invocation by the Strands agent goes through authorization step, maintaining security while enabling powerful functionality.

Gateway 的实现严格遵循 (MCP 授权规范)[https://modelcontextprotocol.org/specification/draft/basic/authorization]，确保强大的安全性和访问控制。这意味着 Strands 代理的每次工具调用都要经过授权步骤，在启用强大功能的同时保持安全性。

For example, when the Strands agent needs to access MCP tools, it first calls ListTools to discover available tools, then uses InvokeTools to execute specific actions. The Gateway handles all the necessary security validations, protocol translations, and service interactions, making the entire process seamless and secure.

例如，当 Strands 代理需要访问 MCP 工具时，它首先调用 ListTools 来发现可用工具，然后使用 InvokeTools 来执行特定操作。Gateway 处理所有必要的安全验证、协议转换和服务交互，使整个过程无缝且安全。

This architectural approach means that any client or SDK that implements the MCP specification can interact with AWS services through the Gateway, making it a versatile and future-proof solution for AI agent integrations.

这种架构方法意味着任何实现 MCP 规范的客户端或 SDK 都可以通过 Gateway 与 AWS 服务交互，使其成为 AI 代理集成的通用且面向未来的解决方案。

![Strands agent calling Gateway](images/strands-lambda-gateway.png)

# Request the access token from Amazon Cognito for inbound authorization

# 从 Amazon Cognito 请求访问令牌用于入站授权

In [ ]:
import time
time.sleep(10)

In [ ]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes")
token_response = utils.get_token(user_pool_id, client_id, client_secret,scopeString,REGION)
token = token_response["access_token"]
print("Token response:", token)

# Strands agent calling MCP tools of AWS Lambda using Bedrock AgentCore Gateway

# Strands 代理使用 Bedrock AgentCore Gateway 调用 AWS Lambda 的 MCP 工具

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL,headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)

## The IAM credentials configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
from strands import Agent
import logging


# Configure the root strands logger. Change it to DEBUG if you are debugging the issue.
logging.getLogger("strands").setLevel(logging.INFO)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

with client:
    # Call the listTools 
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(model=yourmodel,tools=tools) ## you can replace with any model you like
    print(f"Tools loaded in the agent are {agent.tool_names}")
    # print(f"Tools configuration in the agent are {agent.tool_config}")
    # Invoke the agent with the sample prompt. This will only invoke  MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi , can you list all tools available to you")
    # Invoke the agent with sample prompt, invoke the tool and display the response
    agent("Check the order status for order id 123 and show me the exact response from the tool")
    # Call the MCP tool explicitly. The MCP Tool name and arguments must match with your AWS Lambda function or the OpenAPI/Smithy API
    result = client.call_tool_sync(
    tool_use_id="get-order-id-123-call-1", # You can replace this with unique identifier. 
    name=targetname+"___get_order_tool", # This is the tool name based on AWS Lambda target types. This will change based on the target name
    arguments={"orderId": "123"}
    )
    # Print the MCP Tool response
    print(f"Tool Call result: {result['content'][0]['text']}")


**Issue: if you get below error while executing below cell, it indicates incompatibily between pydantic and pydantic-core versions.**

**问题：如果在执行以下单元格时出现以下错误，则表示 pydantic 和 pydantic-core 版本不兼容。**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```

**How to resolve?**

**如何解决？**

You will need to make sure you have pydantic==2.7.2 and pydantic-core 2.27.2 that are both compatible. Restart the kernel once done.

您需要确保安装了兼容的 pydantic==2.7.2 和 pydantic-core 2.27.2。完成后重新启动内核。

# Clean up

# 清理

Additional resources are also created like IAM role, IAM Policies, Credentials provider, AWS Lambda functions, Cognito user pools, s3 buckets that you might need to manually delete as part of the clean up. This depends on the example you run.

还会创建额外的资源，如 IAM 角色、IAM 策略、凭证提供程序、AWS Lambda 函数、Cognito 用户池、S3 存储桶等，您可能需要作为清理的一部分手动删除这些资源。这取决于您运行的示例。

## Delete the gateway (Optional)

## 删除网关（可选）

In [ ]:
import utils
utils.delete_gateway(gateway_client,gatewayID)